# 13 Dashboard: price check
What the dashboard's **Price check** form needs from the data, saved as small CSVs in `data/dashboard/`:
- the choices for every dropdown, most common first
- every (manufacturer, model) with its typical type, drive, fuel, transmission and cylinders, to pre-fill the form
- the median location of each state's listings (the form asks for the state; the model uses lat / long)
- the median description length, year and mileage, for the fields the form doesn't ask for or starts with

Reads `data/vehicles_features.csv` (all the cleaned listings, the same rows the final model was trained on).

In [1]:
import numpy as np
import pandas as pd
from carpricepredictor.shared import DASHBOARD_DIR, FEATURES_CSV

df = pd.read_csv(FEATURES_CSV)
df.shape

(205385, 33)

## Cylinders as the form shows them
`cylinders_num` is the number from "6 cylinders"; engines like rotary are NaN and electric cars are 0 (see `04_modeling.ipynb`). The form offers the numbers plus "other"; an electric car gets 0 from its fuel, whatever is picked.

In [2]:
df["cylinders"] = df["cylinders_num"].map(lambda n: "other" if pd.isna(n) or n == 0 else str(int(n)))
df["cylinders"].value_counts()

cylinders
4        75580
6        75402
8        50126
5         1666
other     1318
10         668
3          484
12         141
Name: count, dtype: int64

## Dropdown choices
One row per (column, value), with how many listings have it; the form lists them most common first.

In [3]:
FORM_COLS = ["condition", "fuel", "cylinders", "transmission", "drive", "type", "paint_color", "title_status", "state"]
options = pd.concat([
    df[c].value_counts().rename_axis("value").rename("listings").reset_index().assign(column=c) for c in FORM_COLS
])[["column", "value", "listings"]]
options.groupby("column").size()

column
condition        7
cylinders        8
drive            4
fuel             5
paint_color      8
state           51
title_status     6
transmission     3
type            13
dtype: int64

## Models and their typical values
The most common type, drive, fuel, transmission and cylinders of each (manufacturer, model), to pre-fill the form when a model is picked.

In [4]:
PREFILL = ["type", "drive", "fuel", "transmission", "cylinders"]
by_model = df.groupby(["manufacturer", "model"])
models = by_model[PREFILL].agg(lambda s: s.mode().iloc[0]).assign(listings=by_model.size()).reset_index()
models.sort_values("listings", ascending=False).head()

,manufacturer,model,type,drive,fuel,transmission,cylinders,listings
257,ford,f-150,pickup,4wd,gas,automatic,6,6839
144,chevrolet,silverado 1500,pickup,4wd,gas,automatic,8,3536
710,ram,1500,pickup,4wd,gas,automatic,8,3410
455,jeep,wrangler,SUV,4wd,gas,automatic,6,3346
334,honda,accord,sedan,fwd,gas,automatic,4,3301


## State locations and defaults
- **states**: median lat / long of each state's listings (listings with impossible coordinates are NaN and skipped)
- **defaults**: median description length (the form doesn't ask for it), and the median year and mileage the form starts with

In [5]:
states = df.groupby("state")[["lat", "long"]].median().reset_index()
defaults = pd.Series({
    "desc_len": df["desc_len"].median(),
    "year": 2021 - df["age"].median(),
    "odometer": df["odometer"].median(),
}).rename_axis("metric").rename("value").reset_index()
defaults

,metric,value
0,desc_len,914.0
1,year,2012.0
2,odometer,105420.0


## Save

In [6]:
DASHBOARD_DIR.mkdir(exist_ok=True)
out = {"pc_options.csv": options, "pc_models.csv": models, "pc_states.csv": states, "pc_defaults.csv": defaults}
for name, table in out.items():
    table.to_csv(DASHBOARD_DIR / name, index=False)
    print(f"{name:18} {len(table):>5,} rows")

pc_options.csv       105 rows
pc_models.csv        993 rows
pc_states.csv         51 rows
pc_defaults.csv        3 rows
